# Ground state search with an autoregressive RWKV network

This notebook shows how to use `jVMC.nets.RWKVCPM`, an autoregressive wave function based on the RWKV architecture, a recurrent alternative to transformers. We use it on the 1D transverse-field Ising model on a small chain, small enough to compare with exact diagonalisation.

In [1]:
import numpy as np
import jax.numpy as jnp

import jVMC
import jVMC.nets as nets
import jVMC.operator.discrete as op
import jVMC.sampler as sampler
from jVMC.vqs import NQS

Running in single-node mode (JVMC_USE_DISTRIBUTED not set)


/Users/som30090/code/Custom_jVMC/jVMC/solver/util.py:10: JaxMgWarning: Unsupported platform darwin, only Linux is supported. Non-Linux only works for docs.
  import jaxmg
/Users/som30090/code/Custom_jVMC/jVMC/solver/util.py:10: JaxMgWarning: No GPUs found, only use this mode for testing or generating documentation.
  import jaxmg


## Hamiltonian

$$
H = -\sum_{i} \sigma^z_i \sigma^z_{i+1} - g \sum_i \sigma^x_i ,
$$

on a chain of $L = 4$ sites with periodic boundary conditions, at the critical point $g = 1$.

In [2]:
L = 4
g = 1.0

hamiltonian = 0
for i in range(L):
    hamiltonian += -op.SigmaZ(i) * op.SigmaZ((i + 1) % L)
    hamiltonian += -g * op.SigmaX(i)

## Autoregressive RWKV network

An autoregressive network writes the wave function as a product of conditional probabilities,

$$
\psi(s) = \prod_{i} \sqrt{p(s_i \mid s_1, \dots, s_{i-1})}\; e^{i \phi(s)} ,
$$

where each conditional $p(s_i \mid s_{<i})$ is computed by the RWKV layers from the previous sites, and $\phi(s)$ is a phase. Because the conditionals are normalised, $\psi$ is normalised by construction, and configurations can be drawn exactly site by site, without a Markov chain. The network provides this through its `sample()` method.

The main hyperparameters are:

- `l_hil_dim`: local Hilbert space dimension (2 for spins 1/2),
- `patch_size`: number of sites grouped into one token (here single sites),
- `num_layers`, `num_heads`, `embedding_size`: number of RWKV layers, attention heads, and dimension of the token embeddings,
- `hidden_size`: dimension of the features from which the conditional probabilities and the phase are computed,
- `param_dtype`, `compute_dtype`: data types of the parameters and of the computation.

With $2^4 = 16$ basis states we use the `ExactSampler`, which sums over all of them. For larger systems, use `jVMC.sampler.MCSampler`: it detects the `sample()` method of the network (`psi.is_generator`) and draws independent samples directly, without a proposer or thermalisation.

In [3]:
net = nets.RWKVCPM(
    L=L,
    l_hil_dim=2,
    patch_size=1,
    hidden_size=8,
    num_heads=1,
    num_layers=1,
    embedding_size=8,
    param_dtype=jnp.float32,
    compute_dtype=jnp.float32,
)
psi = NQS(net, L, batch_size=2**L, seed=1234)
exact_sampler = sampler.ExactSampler(psi)
print("Number of parameters:", psi.num_parameters)

Number of parameters: 465


## Optimisation with MinSR

As in the ViT example, we use `MinSR`, which solves a linear system whose size is the number of samples (16) instead of the number of parameters (465).

In [4]:
loss_function = jVMC.objective_function.Observable(hamiltonian)
stepper = jVMC.stepper.Euler(1e-1)
solver = jVMC.solver.Pinv(pinv_cutoff=1e-8)
opt = jVMC.optimizer.MinSR(exact_sampler, psi, solver=solver, diagonal_shift=1e-3)

out = opt.ground_state_search(200, loss_function, stepper)

100%|██████████| 200/200 [00:03<00:00, 65.07it/s, E=-5.2263e+00+3.3881e-21j ± 2.3766e-04 (Var = 2.4520e-07)] 

Recorded timings:
    - sampling: 0.000147s
    - compute objective function and gradient: 0.000720s
    - solve: 0.000657s


## Comparison with exact diagonalisation

The `ExactSampler` stores the wave function of the last `sample()` call and does not update it when the parameters change. The last optimisation step updates the parameters after the measurement, so we call `sample()` again before measuring the final energy. The exact ground state energy is obtained by diagonalising the $16 \times 16$ Hamiltonian matrix.

In [5]:
exact_sampler.sample()
energy = exact_sampler(hamiltonian).mean.real.item()
H_dense = hamiltonian.to_dense(exact_sampler.basis)
exact_energy = np.linalg.eigvalsh(H_dense)[0]

print(f"VMC energy:     {energy:.8f}")
print(f"Exact energy:   {exact_energy:.8f}")
print(f"Relative error: {abs((energy - exact_energy) / exact_energy):.1e}")

VMC energy:     -5.22625129
Exact energy:   -5.22625186
Relative error: 1.1e-07
